# GDROM vs NID Reservoir Storage Comparison

## Why storage values differ between the two databases

| Database | Storage field | Definition |
|----------|--------------|------------|
| **GDROM** (`GDROM_Resv.csv`) | `max_historical_Storage_af` | Maximum storage **actually observed** in USGS gauge records |
| **NID** (`NID_All_USA_Over_1km2.shp`) | `maxStorage` | **Design total capacity** (conservation pool + flood pool) |
| **NID** | `normalStor` | Conservation pool at normal operating level only |

For flood-control dams, these can differ by 5–75×.  
This notebook reproduces the matching analysis and highlights the clearest examples.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from difflib import SequenceMatcher
import os

ROOT     = r"C:\Users\anyiw\OneDrive - University of Illinois - Urbana\GIS\2026REU\WSDI"
GDROM_CSV = os.path.join(ROOT, "data", "CSV", "GDROM_Resv.csv")
NID_SHP   = os.path.join(ROOT, "data", "NID_All_US_over1km2", "NID_All_USA_Over_1km2.shp")

gdrom = pd.read_csv(GDROM_CSV)
nid   = gpd.read_file(NID_SHP)
nid["lat"] = nid.geometry.centroid.y
nid["lon"] = nid.geometry.centroid.x

print(f"GDROM: {len(gdrom)} reservoirs")
print(f"NID:   {len(nid)} reservoirs (>1 km²)")

## Step 1 — Name + Coordinate Matching

Match each GDROM dam to the closest NID record using:
- **60% name similarity** (SequenceMatcher)
- **40% proximity** (within 0.5° ≈ 50 km)

Pure name matching fails because different reservoirs can share names (e.g., multiple "Wolf Creek" dams exist). Adding the coordinate constraint ensures we are comparing the same physical structure.

In [ ]:
def match_score(gname, glat, glon, nrow, max_dist=0.5):
    dist = np.sqrt((glat - nrow["lat"])**2 + (glon - nrow["lon"])**2)
    if dist > max_dist:
        return -1, dist
    name_sim = SequenceMatcher(None, gname.lower(), str(nrow["name"]).lower()).ratio()
    prox     = max(0.0, 1.0 - dist / max_dist)
    return 0.6 * name_sim + 0.4 * prox, dist

results = []
for _, grow in gdrom.iterrows():
    gname = str(grow["dam name"])
    glat  = grow["lattitude"]
    glon  = grow["longtitude"]
    gstor = grow["max_historical_Storage_af"]

    best_score, best_nrow, best_dist = -1, None, None
    for _, nrow in nid.iterrows():
        sc, dist = match_score(gname, glat, glon, nrow)
        if sc > best_score:
            best_score, best_nrow, best_dist = sc, nrow, dist

    if best_nrow is not None and gstor > 0 and pd.notna(best_nrow["Str_af_num"]):
        nstor     = best_nrow["Str_af_num"]
        norm_stor = pd.to_numeric(best_nrow["normalStor"], errors="coerce")
        name_sim  = SequenceMatcher(None, gname.lower(),
                                    str(best_nrow["name"]).lower()).ratio()
        ratio     = gstor / nstor if nstor > 0 else np.nan
        results.append({
            "GDROM_name":   gname,
            "NID_name":     best_nrow["name"],
            "name_sim":     round(name_sim, 3),
            "dist_deg":     round(best_dist, 4),
            "GDROM_af":     gstor,
            "NID_max_af":   nstor,
            "NID_norm_af":  norm_stor,
            "primaryPur":   best_nrow["primaryPur"],
            "state":        grow["state"],
            "ratio_G_N":    round(ratio, 3) if not np.isnan(ratio) else np.nan,
        })

df = pd.DataFrame(results)
print(f"Matched {len(df)} GDROM dams to NID counterparts")
print(f"Name similarity >= 0.7 and distance < 0.5 deg: {((df.name_sim>=0.7)&(df.dist_deg<0.5)).sum()} dams")

## Step 2 — Find Reservoirs with Large Storage Discrepancies

Filter for confident matches (name similarity > 0.7, distance < 0.5°) where storage ratio is either > 3× or < 0.33×.

In [ ]:
confident = df[(df["name_sim"] >= 0.7) & (df["dist_deg"] < 0.5) & df["ratio_G_N"].notna()]
discrepant = confident[(confident["ratio_G_N"] > 3) | (confident["ratio_G_N"] < 0.33)].copy()
discrepant["direction"] = discrepant["ratio_G_N"].apply(
    lambda r: "GDROM >> NID" if r > 3 else "NID >> GDROM"
)
discrepant = discrepant.sort_values("ratio_G_N", ascending=False)

display_cols = ["GDROM_name", "NID_name", "name_sim", "dist_deg",
                "GDROM_af", "NID_max_af", "NID_norm_af", "ratio_G_N", "direction", "state"]
print(f"Discrepant matched pairs (ratio >3 or <0.33): {len(discrepant)}")
discrepant[display_cols].reset_index(drop=True)

## Step 3 — Four Key Examples

These are the clearest cases: exact or near-exact name match, same location, but very different storage values.

In [ ]:
key_examples = [
    "Chatfield Dam",
    "Jemez Canyon Dam",
    "Clarence J. Brown Dam",
    "Max Starcke Dam",
]

# Pull the matched rows for these four dams
examples_df = df[df["GDROM_name"].isin(key_examples)].set_index("GDROM_name").loc[key_examples]

# Also fetch NID primaryPur description
pur_map = {
    "1": "Hydroelectric", "2": "Irrigation", "3": "Navigation",
    "4": "Recreation",    "5": "Water Supply", "6": "Tailings",
    "7": "Fish & Wildlife", "8": "Fire Protection", "9": "Flood Control",
    "10": "Other",
}
examples_df["NID_purpose"] = examples_df["primaryPur"].astype(str).map(
    lambda x: pur_map.get(str(int(float(x))) if x not in ("nan","") else "?", x)
)

show = examples_df[["NID_name", "name_sim", "dist_deg",
                     "GDROM_af", "NID_max_af", "NID_norm_af",
                     "ratio_G_N", "NID_purpose", "state"]]
show.index.name = "GDROM_name"
show

## Step 4 — Bar Chart: GDROM vs NID storage for the four examples

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
colors = {"GDROM (observed max)": "#2196F3",
          "NID max (design cap.)": "#F44336",
          "NID normal pool":       "#FF9800"}

for ax, name in zip(axes, key_examples):
    row = examples_df.loc[name]
    bars = {
        "GDROM\n(observed max)":  row["GDROM_af"],
        "NID\n(design cap.)":     row["NID_max_af"],
        "NID\n(normal pool)":     row["NID_norm_af"] if pd.notna(row["NID_norm_af"]) else 0,
    }
    bar_colors = ["#2196F3", "#F44336", "#FF9800"]
    x = range(len(bars))
    rects = ax.bar(x, bars.values(), color=bar_colors, alpha=0.85, width=0.55)
    ax.set_xticks(list(x))
    ax.set_xticklabels(bars.keys(), fontsize=8)
    ax.set_title(f"{name}\n({row['state']})", fontsize=9, fontweight="bold")
    ax.set_ylabel("Storage (acre-feet)" if ax == axes[0] else "")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1e3:.0f}k"))
    for rect in rects:
        h = rect.get_height()
        if h > 0:
            ax.text(rect.get_x() + rect.get_width()/2, h * 1.02,
                    f"{h:,.0f}", ha="center", va="bottom", fontsize=7)
    ax.set_ylim(0, max(bars.values()) * 1.25)

patches = [mpatches.Patch(color=c, label=l, alpha=0.85)
           for l, c in zip(["GDROM (observed max)", "NID max (design cap.)", "NID normal pool"],
                           ["#2196F3", "#F44336", "#FF9800"])]
fig.legend(handles=patches, loc="lower center", ncol=3, fontsize=9, frameon=False,
           bbox_to_anchor=(0.5, -0.05))
fig.suptitle("GDROM vs NID Storage — Same Dam, Different Definitions", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## Step 5 — Scatter Plot: All confident matches

Each point is one GDROM dam matched to a NID dam (name similarity ≥ 0.7, distance < 0.5°).  
Points on the diagonal = agreement. Points above = NID reports more. Points below = GDROM reports more.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

conf = confident.copy()
conf["log_G"] = np.log10(conf["GDROM_af"].clip(lower=1))
conf["log_N"] = np.log10(conf["NID_max_af"].clip(lower=1))

# Color by ratio
colors_scatter = conf["ratio_G_N"].apply(
    lambda r: "#F44336" if r > 3 else ("#FF9800" if r < 0.33 else "#4CAF50")
)
ax.scatter(conf["log_N"], conf["log_G"], c=colors_scatter, alpha=0.6, s=25, linewidths=0)

# Diagonal (perfect agreement)
lims = [3, 8]
ax.plot(lims, lims, "k--", linewidth=1, label="1:1 line")
ax.fill_between(lims, [l - np.log10(3) for l in lims],
                       [l + np.log10(3) for l in lims],
                color="grey", alpha=0.1, label="within 3x")

# Annotate the 4 key examples
for name in key_examples:
    row = conf[conf["GDROM_name"] == name]
    if len(row):
        ax.annotate(name, xy=(row["log_N"].values[0], row["log_G"].values[0]),
                    xytext=(8, 5), fontsize=7.5,
                    arrowprops=dict(arrowstyle="->", color="black", lw=0.8),
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))

ax.set_xlabel("log₁₀  NID maxStorage  (ac·ft)", fontsize=11)
ax.set_ylabel("log₁₀  GDROM max_historical_Storage  (ac·ft)", fontsize=11)
ax.set_title("GDROM vs NID Storage — All Confident Matches\n"
             "(green = within 3×, red = GDROM>>NID, orange = NID>>GDROM)", fontsize=11)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xticks(range(3, 9)); ax.set_yticks(range(3, 9))
ax.set_xticklabels([f"10^{i}\n({10**i:,.0f})" for i in range(3, 9)], fontsize=8)
ax.set_yticklabels([f"10^{i}" for i in range(3, 9)], fontsize=8)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

n_agree = ((conf["ratio_G_N"] >= 1/3) & (conf["ratio_G_N"] <= 3)).sum()
print(f"Within 3x: {n_agree}/{len(conf)} ({100*n_agree/len(conf):.0f}%) of confident matches")

## Summary

| Dam | State | GDROM (ac·ft) | NID max (ac·ft) | NID normal (ac·ft) | Ratio | Explanation |
|-----|-------|--------------|----------------|-------------------|-------|-------------|
| **Chatfield Dam** | CO | 56,986 | 355,000 | 20,000 | 0.16 | Flood-control dam; GDROM observed max between normal and flood pool |
| **Jemez Canyon Dam** | NM | 3,501 | 264,700 | 29,712 | 0.01 | Dry detention dam; nearly always empty |
| **Clarence J. Brown Dam** | OH | 603,909 | 63,700 | 36,900 | 9.5 | GDROM >> NID; possible data discrepancy |
| **Max Starcke Dam** | TX | 238,090 | 8,760 | 8,760 | 27 | GDROM >> NID; possible data discrepancy |

**Key takeaway:**  
For flood-control dams (NID `primaryPur = 9`), NID `maxStorage` is the design capacity and GDROM records actual observed storage — these are measuring different things. Using `normalStor` from NID is often a better comparison point for GDROM's observed values.